# 05 · Embeddings: turning text into vectors
Level L1 · Part 1 · builds on lesson 04

## Goal
By the end of this lesson you can embed recipe descriptions with a small sentence-embedding model
and show, with printed similarity scores, that recipes with similar meanings land close together.

## The idea
In lessons 02–04 *we* gave each recipe two numbers, `[sweet, spicy]`. That cannot grow: nobody
hand-scores 50 recipes on hundreds of features. An **embedding model** is a neural network that
reads a piece of text and returns a fixed-length vector, called an **embedding**, placing texts
with similar meanings near each other. The model learned where to put things from millions of
example sentences, so we never choose the features. The price: the vector has hundreds of
numbers (its **dimensionality**), and none of them has a name like "sweet".
Think of a librarian who has read every book and shelves them by topic without labels:
you can't read the shelf numbers, but neighbouring books are about similar things.

## Picture

```mermaid
flowchart LR
    T["'Tomato soup. Simmer tomatoes,<br/>onion and garlic...'"] --> M["embedding model<br/>all-MiniLM-L6-v2"]
    Q["query: 'something warm<br/>for a cold day'"] --> M
    M --> V["384 numbers per text<br/>(length 1)"]
    V --> K["knn from lesson 04<br/>(cosine)"]
    K --> R["top-3 recipes"]
```

The only new box is the model. Everything after it is lesson 04's `knn`, now on 384 numbers instead of 2.

## Step by step

### 1. Fifty recipes
Five recipes were enough to see the idea; to see embeddings group things we need more. The course
ships 50 recipes in `data/recipes_50.csv`. The first five are our old friends from lesson 01, with the
same descriptions. The other columns (cuisine, vegetarian, minutes) wait for lesson 09.

In [1]:
import numpy as np
import pandas as pd

recipes = pd.read_csv("../data/recipes_50.csv")
names = recipes["title"].tolist()
texts = (recipes["title"] + ". " + recipes["description"]).tolist()   # what we embed
print(recipes.shape)
print(recipes[["title", "cuisine", "vegetarian", "minutes"]].head(6).to_string(index=False))
print("\nfirst text to embed:", texts[0])

(50, 6)
             title  cuisine  vegetarian  minutes
       Tomato soup  Italian        True       35
       Mango salsa  Mexican        True       10
Chocolate brownies American        True       45
      Caesar salad  Italian        True       15
      Lemon sorbet   French        True       20
        Minestrone  Italian        True       50

first text to embed: Tomato soup. Simmer tomatoes, onion and garlic, blend until smooth and serve hot with crusty bread.


Each recipe becomes one string, title plus description, because both carry meaning.

### 2. Load a model and embed one sentence
We use `sentence-transformers/all-MiniLM-L6-v2`: small (about 90 MB), fast on a laptop CPU, and
not gated, so no account is needed. The first run downloads it into your Hugging Face cache, so
this cell is tagged `needs-download`. `encode_query` turns one text into one vector. The first
three lines only silence download chatter.

In [2]:
import os, warnings
os.environ["TQDM_DISABLE"] = "1"                  # hide progress bars
os.environ["HF_HUB_VERBOSITY"] = "error"          # hide the "no HF_TOKEN" notice (no token is needed)
warnings.filterwarnings("ignore", message="IProgress")
from sentence_transformers import SentenceTransformer

MINILM = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MINILM, device="cpu")
v = model.encode_query("something warm for a cold day")
print("embedding dimension:", model.get_embedding_dimension())
print("shape:", v.shape, " dtype:", v.dtype)
print("first 6 numbers:", np.round(v[:6], 3))
print("length:", round(float(np.linalg.norm(v)), 3))

embedding dimension: 384
shape: (384,)  dtype: float32
first 6 numbers: [-0.1    0.05   0.043  0.132  0.038 -0.012]
length: 1.0


One sentence in, 384 numbers out, stored as `float32`. Its length is 1.0: this model ends with a
normalisation step (lesson 03), so every embedding is a **unit vector**. Remember that for step 7.

### 3. Embed all 50 recipes and some queries, then save them
`encode_document` embeds the recipes, `encode_query` the questions. For this model the two do the
same thing; some models add a different instruction to each, so using the right one is a good
habit. We save the result to `data/`, so later cells (and lesson 06) do not need the model.

In [3]:
QUERIES = ["something warm for a cold day", "a cold dessert for summer",
           "very spicy food", "something to eat in the morning", "quick vegetarian dinner"]
doc_emb = model.encode_document(texts)          # one row per recipe
query_emb = model.encode_query(QUERIES)         # one row per query
np.savez("../data/recipe_embeddings_minilm.npz", model=MINILM, names=names, texts=texts,
         doc_emb=doc_emb, queries=QUERIES, query_emb=query_emb)
print("recipes:", doc_emb.shape, " queries:", query_emb.shape, " -> saved")

recipes: (50, 384)  queries: (5, 384)  -> saved


### 4. Load the saved embeddings
This cell always runs. If you ran the two cells above, it reads the file you just wrote; if they
were skipped (no network), it reads the copy that ships with the course, made by the same code.
Either way the rest of the lesson works without the model.

In [4]:
saved = np.load("../data/recipe_embeddings_minilm.npz")
doc_emb, query_emb = saved["doc_emb"], saved["query_emb"]
QUERIES = saved["queries"].tolist()
assert saved["names"].tolist() == names, "embeddings and CSV are out of step"
print("model:  ", saved["model"])
print("recipes:", doc_emb.shape, doc_emb.dtype)
print("queries:", query_emb.shape)

model:   sentence-transformers/all-MiniLM-L6-v2
recipes: (50, 384) float32
queries: (5, 384)


The **shape** `(50, 384)` reads as lesson 02 taught: 50 vectors of 384 dimensions each. The
`assert` checks that row *i* of the matrix is still recipe *i* of the CSV.

### 5. What do 384 numbers look like?
Print the first eight numbers of three recipes, and every recipe's length.

In [5]:
for title in ["Tomato soup", "Minestrone", "Lemon sorbet"]:
    print(f"{title:13}", np.round(doc_emb[names.index(title)][:8], 3))

lengths = np.linalg.norm(doc_emb, axis=1)
print(f"\nlengths: min {lengths.min():.4f}  max {lengths.max():.4f}")

Tomato soup   [-0.144  0.015  0.034  0.041 -0.108  0.008 -0.054 -0.113]
Minestrone    [-0.108  0.023  0.017  0.028 -0.058 -0.037 -0.017 -0.052]
Lemon sorbet  [ 0.028  0.006 -0.008  0.076 -0.008  0.047  0.029 -0.025]

lengths: min 1.0000  max 1.0000


Small positive and negative numbers, and no column you can name. Unlike `[sweet, spicy]`, no single
dimension means anything on its own: meaning lives in the *direction* of the whole vector. All 50
lengths are 1.0000, so these are unit vectors.

### 6. Similar meanings land close together
Cosine similarity (lesson 03) between Tomato soup and four other recipes. For unit vectors it is
just the dot product.

In [6]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

soup = doc_emb[names.index("Tomato soup")]
for other in ["Minestrone", "Gazpacho", "Caesar salad", "Lemon sorbet", "Chocolate brownies"]:
    print(f"Tomato soup vs {other:19} {cosine(soup, doc_emb[names.index(other)]):.3f}")

Tomato soup vs Minestrone          0.687
Tomato soup vs Gazpacho            0.625
Tomato soup vs Caesar salad        0.516
Tomato soup vs Lemon sorbet        0.291
Tomato soup vs Chocolate brownies  0.253


Minestrone (another tomato soup) and Gazpacho score highest; the desserts lowest. Nobody told
the model "soup" is a feature: it placed them close from the text alone. Note the numbers: even
the desserts score about 0.25–0.3, not 0. With this model, what matters is the **ranking**,
not the raw value.

### 7. Search with lesson 04's `knn`
Lesson 04's final `knn` and `top_k_smallest`, re-declared unchanged. It never knew vectors had to
be 2-D, so it works on 384 numbers as it is.

In [7]:
def top_k_smallest(scores, k):
    k = min(k, len(scores))
    part = np.argpartition(scores, k - 1)[:k]
    return part[np.argsort(scores[part], kind="stable")]

def knn(query, vectors, k, metric="euclidean"):
    q = np.asarray(query, dtype=np.float32)
    if metric == "euclidean":
        scores = np.linalg.norm(vectors - q, axis=1)
        rows = top_k_smallest(scores, k)                      # smaller = closer
    elif metric in ("dot", "cosine"):
        scores = vectors @ q
        if metric == "cosine":
            scores = scores / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(q))
        rows = top_k_smallest(-scores, k)                     # bigger = closer
    else:
        raise ValueError(f"unknown metric {metric!r}")
    return rows, scores[rows]

Now run all five saved queries. Remember lesson 01: keyword search found *nothing* for
"something warm for a cold day", because no recipe contains those words.

In [8]:
def show(rows, scores, label):
    print(label)
    for rank, (i, s) in enumerate(zip(rows, scores, strict=True), start=1):
        print(f"  {rank}. {names[i]:22} {s:.3f}")

for text, q in zip(QUERIES, query_emb, strict=True):
    show(*knn(q, doc_emb, k=3, metric="cosine"), label=f"'{text}':")

'something warm for a cold day':
  1. Chicken noodle soup    0.444
  2. Gazpacho               0.399
  3. Roast chicken          0.377
'a cold dessert for summer':
  1. Lemon sorbet           0.603
  2. Apple pie              0.495
  3. Vanilla ice cream      0.473
'very spicy food':
  1. Spicy tofu stir-fry    0.634
  2. Vindaloo               0.558
  3. Chili con carne        0.552
'something to eat in the morning':
  1. Full English breakfast 0.593
  2. Overnight oats         0.525
  3. Chicken noodle soup    0.491
'quick vegetarian dinner':
  1. Beef stew              0.559
  2. Full English breakfast 0.515
  3. Guacamole              0.484


Most answers are sensible with zero shared words: Chicken noodle soup "for chilly evenings" tops
the warm query, Lemon sorbet the cold dessert, a stir-fry, a curry and a chili the spicy one, breakfasts the
morning one. But read closely. Gazpacho, a *chilled* soup, is second for "warm": the model sees
"soup" and "hot day" and misses the opposite temperature. And "quick vegetarian dinner" returns
Beef stew, a 3-hour meat dish. Embeddings capture *topic*; hard facts like "vegetarian" or "under
30 minutes" are better as filters (lesson 11).

### 8. Unit vectors: cosine, dot and Euclidean agree
Because every embedding has length 1, lesson 03's rule applies: dot product equals cosine, and
Euclidean distance gives the same order too. Check it on the first query.

In [9]:
q = query_emb[0]
for metric in ["cosine", "dot", "euclidean"]:
    rows, scores = knn(q, doc_emb, k=5, metric=metric)
    print(f"{metric:9}", [names[i] for i in rows], np.round(scores, 3))

cosine    ['Chicken noodle soup', 'Gazpacho', 'Roast chicken', 'Ramen', 'Lemon sorbet'] [0.444 0.399 0.377 0.375 0.348]
dot       ['Chicken noodle soup', 'Gazpacho', 'Roast chicken', 'Ramen', 'Lemon sorbet'] [0.444 0.399 0.377 0.375 0.348]
euclidean ['Chicken noodle soup', 'Gazpacho', 'Roast chicken', 'Ramen', 'Lemon sorbet'] [1.054 1.096 1.117 1.118 1.142]


Same five recipes in the same order, and the cosine and dot scores are identical. That is why many
systems normalise once at insert time and then use the cheaper dot product (lesson 08 picks a
distance for a collection).

### 9. A second model sees things differently
Embeddings belong to the model that made them. `BAAI/bge-small-en-v1.5` also returns 384 numbers.
Embed the same texts with it (another download), save, and compare.

In [10]:
BGE = "BAAI/bge-small-en-v1.5"
bge = SentenceTransformer(BGE, device="cpu")
np.savez("../data/recipe_embeddings_bge-small.npz", model=BGE, names=names,
         doc_emb=bge.encode_document(texts), queries=QUERIES, query_emb=bge.encode_query(QUERIES))
print("embedding dimension:", bge.get_embedding_dimension(), "-> saved")

embedding dimension: 384 -> saved


As in step 4, this cell always runs and loads the saved file.

In [11]:
saved_bge = np.load("../data/recipe_embeddings_bge-small.npz")
bge_doc, bge_query = saved_bge["doc_emb"], saved_bge["query_emb"]
print("bge-small:", bge_doc.shape)
for i in [0, 4]:
    show(*knn(query_emb[i], doc_emb, k=3, metric="cosine"), label=f"MiniLM    '{QUERIES[i]}':")
    show(*knn(bge_query[i], bge_doc, k=3, metric="cosine"), label=f"bge-small '{QUERIES[i]}':")

bge-small: (50, 384)
MiniLM    'something warm for a cold day':
  1. Chicken noodle soup    0.444
  2. Gazpacho               0.399
  3. Roast chicken          0.377
bge-small 'something warm for a cold day':
  1. Gazpacho               0.659
  2. Lemon sorbet           0.646
  3. Chicken noodle soup    0.637
MiniLM    'quick vegetarian dinner':
  1. Beef stew              0.559
  2. Full English breakfast 0.515
  3. Guacamole              0.484
bge-small 'quick vegetarian dinner':
  1. Caprese salad          0.719
  2. Hummus                 0.699
  3. Sushi rolls            0.688


Different models, different neighbours, and very different score ranges: bge-small's scores sit
around 0.64–0.72 even for poor matches (Lemon sorbet for "warm"), MiniLM's around 0.38–0.56.
bge-small does worse on "warm" (Gazpacho and Lemon sorbet first) and better on "vegetarian". Neither model is "right"; lesson 16
shows how to measure which one serves your queries better, and lesson 18 how to choose one.

## What just happened
- We turned 50 recipe texts into a `(50, 384)` matrix with one model call, with no hand-made features (steps 1–4).
- The numbers have no names, but every vector has length 1.0000, so direction carries the meaning (step 5).
- Tomato soup scored highest against Minestrone and Gazpacho and lowest against desserts: similar meanings land close (step 6).
- Lesson 04's `knn` worked unchanged on 384 dimensions and answered "something warm for a cold day" with Chicken noodle soup, a query keyword search could not answer (step 7).
- It also made mistakes (Gazpacho for "warm", Beef stew for "vegetarian"): embeddings capture topic, not every fact (step 7).
- Cosine, dot and Euclidean gave the same order on unit vectors (step 8), and a second model gave different neighbours and scores (step 9).

## Common mistakes

**1. Searching embeddings with a hand-made query.** Our old `[sweet, spicy]` query has 2 numbers;
the recipes now have 384. NumPy refuses to compare them:

In [12]:
try:
    knn([0.3, 0.4], doc_emb, k=3)
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

ValueError: operands could not be broadcast together with shapes (50,384) (2,) 


It prints `ValueError: operands could not be broadcast together with shapes (50,384) (2,)`. Fix:
a query must go through the *same model* as the recipes. Here, use the saved query embedding
(with the model loaded: `model.encode_query("...")`).

In [13]:
rows, scores = knn(query_emb[0], doc_emb, k=3, metric="cosine")
print([names[i] for i in rows])

['Chicken noodle soup', 'Gazpacho', 'Roast chicken']


**2. Mixing two models.** Worse than a shape error: bge-small and MiniLM both give 384 numbers, so
a bge query against MiniLM recipes runs without complaint. The results are meaningless, because
the two models arrange their 384 dimensions in unrelated ways. Ask for "a cold dessert for summer":

In [14]:
try:
    rows, scores = knn(bge_query[1], doc_emb, k=3, metric="cosine")     # bge query, MiniLM recipes
    show(rows, scores, label="mixed models, 'a cold dessert for summer':")
    assert names[rows[0]] == "Lemon sorbet", "mixed-model search did not find the obvious dessert"
except Exception as err:  # this mistake is on purpose: show the error, keep the notebook running
    print(f"{type(err).__name__}: {err}")

mixed models, 'a cold dessert for summer':
  1. Pancakes               0.250
  2. Caprese salad          0.241
  3. Shakshuka              0.232
AssertionError: mixed-model search did not find the obvious dessert


Low scores and unrelated recipes, and the check prints
`AssertionError: mixed-model search did not find the obvious dessert`. Fix: store the model name
next to the vectors (our `.npz` files do, in `model`) and check it before searching.

In [15]:
def search(query_vec, query_model, k=3):
    assert query_model == str(saved["model"]), f"query from {query_model}, recipes from {saved['model']}"
    return knn(query_vec, doc_emb, k=k, metric="cosine")

show(*search(query_emb[1], str(saved["model"])), label="same model, 'a cold dessert for summer':")

same model, 'a cold dessert for summer':
  1. Lemon sorbet           0.603
  2. Apple pie              0.495
  3. Vanilla ice cream      0.473


## Try it
1. **More like Mango salsa.** Using the saved MiniLM embeddings, find the 3 recipes most similar to
   Mango salsa, leaving out Mango salsa itself (lesson 04, mistake 3). Then do the same with the
   bge-small embeddings. How many of the three do the two models agree on?
2. **The odd one out.** Compute all 50 × 50 cosine similarities in one line
   (`doc_emb @ doc_emb.T`, valid because the vectors have length 1). Ignoring each recipe's
   similarity to itself, which recipe's *nearest* neighbour is the farthest away?
   Hint: `np.fill_diagonal(sims, -1)` hides the diagonal.

In [16]:
# your code here

## Key terms
- **embedding** — a vector produced by a model to represent a piece of text (or image, audio...), placed so that similar meanings are close together.
- **embedding model** — a trained neural network that turns input into embeddings of a fixed size; vectors from different models cannot be compared.
- **dimensionality** — how many numbers each vector has; 384 for all-MiniLM-L6-v2 and bge-small-en-v1.5, 2 for our hand-made `[sweet, spicy]`.
- **sentence-transformers** — the Python library used here to load embedding models and call `encode_query` / `encode_document`.